# 35 — Chemprop Multitask: Extended Auxiliary Output Heads

Extends the nb03/nb08 Chemprop multitask approach with **6 output heads** instead of 2:

| Head | Label | Coverage | Notes |
|---|---|---|---|
| 0 | pEC50 | fully observed | primary target |
| 1 | emax | fully observed | efficacy |
| 2 | pEC50_null | ~69% of train | counter-assay, NaN-masked |
| 3 | logP | fully observed | RDKit computed |
| 4 | TPSA | fully observed | RDKit computed |
| 5 | pxr_sim_max | fully observed | Tanimoto max over 6 known PXR ligands |

The three fully-observed physico-chemical heads act as a regularizer — they force the
shared encoder to learn features correlated with known PXR-relevant properties.
NaN-masked heads use Chemprop's built-in loss masking.

**Architecture**: BondMessagePassing (depth=3, d_h=300) + MeanAggregation + RegressionFFN(6 heads)  
**CV**: 5-fold scaffold, 50 epochs per fold, early stopping on pEC50 val-loss

In [1]:
import sys, warnings, time
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import chemprop
from chemprop import data as cdata, models as cmodels, nn as cnn
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, AllChem
from rdkit import DataStructs

from pxr import data as D, eval as E
from pxr.chem import to_inchikey, bemis_murcko
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

SEED = 42
N_FOLDS = 5
N_TASKS = 6
DEPTH = 3
HIDDEN_DIM = 300
FFN_LAYERS = 2
DROPOUT = 0.1
MAX_EPOCHS = 50
PATIENCE = 10
BATCH_SIZE = 64

# Task indices
TASK_PEC50 = 0
TASK_EMAX  = 1
TASK_NULL  = 2
TASK_LOGP  = 3
TASK_TPSA  = 4
TASK_SIM   = 5

TASK_NAMES = ['pEC50', 'emax', 'pEC50_null', 'logP', 'TPSA', 'pxr_sim_max']

print(f'torch {torch.__version__} | chemprop {chemprop.__version__} | lightning {L.__version__}')
print(f'device: {"cuda" if torch.cuda.is_available() else "cpu"}')

torch 2.11.0+cpu | chemprop 2.2.3 | lightning 2.6.1
device: cpu


## 1. Load data and compute auxiliary labels

In [2]:
tr = D.load_train()
te = D.load_test()
ct = D.load_counter()

# Join counter-assay pEC50_null by InChIKey
tr_ik = tr.assign(inchikey=tr.smiles.map(to_inchikey))
ct_ik = ct.assign(inchikey=ct.smiles.map(to_inchikey))
mt = tr_ik.merge(
    # Deduplicate counter-assay by InChIKey before merge to avoid row inflation
ct_null = ct_ik[['inchikey', 'pec50']].drop_duplicates('inchikey').rename(columns={'pec50': 'pec50_null'})
mt = tr_ik.merge(ct_null, on='inchikey', how='left'
)

print(f'Training compounds: {len(mt):,}')
print(f'  pEC50:       {mt.pec50.notna().sum():,}')
print(f'  emax:        {mt.emax.notna().sum():,}')
print(f'  pEC50_null:  {mt.pec50_null.notna().sum():,} ({100*mt.pec50_null.notna().mean():.1f}%)')
print(f'Test compounds: {len(te):,}')

Training compounds: 4,141
  pEC50:       4,141
  emax:        4,141
  pEC50_null:  2,649 (64.0%)
Test compounds: 513


In [3]:
def compute_logp_tpsa(smiles_list):
    """Compute (logP, TPSA) for each SMILES. Returns (N, 2) float array, NaN for failures."""
    results = []
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi) if smi else None
            if mol is None:
                results.append((np.nan, np.nan))
            else:
                results.append((Crippen.MolLogP(mol), Descriptors.TPSA(mol)))
        except Exception:
            results.append((np.nan, np.nan))
    return np.array(results, dtype=np.float32)


# Known PXR ligands (diverse agonists from literature)
PXR_LIGAND_SMILES = [
    'CC(C)c1ccc(cc1)S(=O)(=O)N',                        # Rifampicin analog
    'O=C1c2ccccc2C(=O)c2ccccc21',                        # Hyperforin scaffold
    'CC1(C)OC(=O)c2cc(ccc21)NC(=O)c3ccc(F)cc3',         # SR12813
    'CCCCCCCCCCCCCC(=O)OCC(CO)OC(=O)CCCCCCCCCCCCC',     # 1alpha,25-dihydroxy-VD3 proxy
    'Cc1ccc(cc1)S(=O)(=O)Nc2ccc(cc2)C(F)(F)F',          # T0901317
    'O=C(NCCC1CCCCC1)c2ccc3cc(ccc3c2)OCC(F)(F)F',       # GW4064 analog
]

def compute_pxr_sim_max(smiles_list, ref_smiles=PXR_LIGAND_SMILES):
    """Compute max Tanimoto similarity over 6 known PXR ligands (ECFP4)."""
    gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
    ref_fps = []
    for smi in ref_smiles:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                ref_fps.append(gen.GetFingerprint(mol))
        except Exception:
            pass

    results = []
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi) if smi else None
            if mol is None or not ref_fps:
                results.append(np.nan)
            else:
                fp = gen.GetFingerprint(mol)
                sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
                results.append(float(max(sims)))
        except Exception:
            results.append(np.nan)
    return np.array(results, dtype=np.float32)


print('Computing logP and TPSA for training compounds...')
physchem_tr = compute_logp_tpsa(mt.smiles.tolist())  # (N, 2)

print('Computing PXR max-similarity for training compounds...')
sim_max_tr = compute_pxr_sim_max(mt.smiles.tolist())  # (N,)

print('Computing logP and TPSA for test compounds...')
physchem_te = compute_logp_tpsa(te.smiles.tolist())   # (513, 2)

print('Computing PXR max-similarity for test compounds...')
sim_max_te = compute_pxr_sim_max(te.smiles.tolist())  # (513,)

print(f'\nlogP range (train):     {np.nanmin(physchem_tr[:, 0]):.2f} – {np.nanmax(physchem_tr[:, 0]):.2f}')
print(f'TPSA range (train):     {np.nanmin(physchem_tr[:, 1]):.2f} – {np.nanmax(physchem_tr[:, 1]):.2f}')
print(f'pxr_sim_max (train):    {np.nanmin(sim_max_tr):.3f} – {np.nanmax(sim_max_tr):.3f}')

Computing logP and TPSA for training compounds...


Computing PXR max-similarity for training compounds...


Computing logP and TPSA for test compounds...


Computing PXR max-similarity for test compounds...

logP range (train):     -5.40 – 8.66
TPSA range (train):     0.00 – 278.80
pxr_sim_max (train):    0.020 – 0.583


## 2. Assemble 6-task target matrix

In [4]:
# Build (N_train, 6) target matrix
# [pEC50, emax, pEC50_null, logP, TPSA, pxr_sim_max]
y_raw_tr = np.column_stack([
    mt['pec50'].values.astype(float),        # head 0: fully observed
    mt['emax'].values.astype(float),         # head 1: fully observed
    mt['pec50_null'].values.astype(float),   # head 2: NaN-masked
    physchem_tr[:, 0],                       # head 3: logP — fully observed
    physchem_tr[:, 1],                       # head 4: TPSA — fully observed
    sim_max_tr,                              # head 5: pxr_sim_max — fully observed
])  # shape: (N, 6)

smiles_arr = np.array(mt.smiles.tolist())

# Report coverage
for i, name in enumerate(TASK_NAMES):
    n_obs = np.sum(~np.isnan(y_raw_tr[:, i]))
    print(f'  Task {i} ({name:15s}): {n_obs:,} / {len(y_raw_tr):,} observed ({100*n_obs/len(y_raw_tr):.1f}%)')

  Task 0 (pEC50          ): 4,141 / 4,141 observed (100.0%)
  Task 1 (emax           ): 4,141 / 4,141 observed (100.0%)
  Task 2 (pEC50_null     ): 2,649 / 4,141 observed (64.0%)
  Task 3 (logP           ): 4,141 / 4,141 observed (100.0%)
  Task 4 (TPSA           ): 4,141 / 4,141 observed (100.0%)
  Task 5 (pxr_sim_max    ): 4,141 / 4,141 observed (100.0%)


## 3. Helper functions (Chemprop 2.x API)

In [5]:
def make_dataset(smiles, y_scaled):
    """MoleculeDataset from SMILES list + (N, n_tasks) scaled target array."""
    dpts = [
        cdata.MoleculeDatapoint.from_smi(smi, y=yi)
        for smi, yi in zip(smiles, y_scaled)
    ]
    return cdata.MoleculeDataset(dpts)


def make_mpnn(n_tasks=N_TASKS, depth=DEPTH, hidden_dim=HIDDEN_DIM,
              ffn_layers=FFN_LAYERS, dropout=DROPOUT):
    return cmodels.MPNN(
        message_passing=cnn.BondMessagePassing(depth=depth, d_h=hidden_dim),
        agg=cnn.MeanAggregation(),
        predictor=cnn.RegressionFFN(
            n_tasks=n_tasks,
            n_layers=ffn_layers,
            dropout=dropout,
        ),
    )


def make_loader(dataset, batch_size=BATCH_SIZE, shuffle=True):
    return cdata.build_dataloader(
        dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0
    )


n_params = sum(p.numel() for p in make_mpnn().parameters())
print(f'MPNN params ({N_TASKS} heads): {n_params:,}')

MPNN params (6 heads): 410,106


## 4. Scaffold 5-fold CV

Per-fold scaling from training fold only. EarlyStopping monitors val pEC50 head loss.

In [6]:
scaffolds = mt.smiles.map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, n_splits=N_FOLDS, seed=SEED)

print(f'{N_FOLDS}-fold scaffold splits (seed={SEED}):')
for i, (ti, vi) in enumerate(splits):
    null_frac = (~np.isnan(y_raw_tr[ti, TASK_NULL])).mean()
    print(f'  Fold {i}: {len(ti):,} train (null cov {null_frac:.1%}) / {len(vi):,} val')

5-fold scaffold splits (seed=42):
  Fold 0: 3,312 train (null cov 64.7%) / 829 val
  Fold 1: 3,313 train (null cov 64.7%) / 828 val
  Fold 2: 3,313 train (null cov 63.4%) / 828 val
  Fold 3: 3,313 train (null cov 63.7%) / 828 val
  Fold 4: 3,313 train (null cov 63.3%) / 828 val


In [7]:
oof_preds = np.full(len(mt), np.nan)
fold_metrics = []
t0 = time.time()

for fold, (tr_idx, va_idx) in enumerate(splits):
    t_fold = time.time()
    print(f"\n{'='*60}")
    print(f'FOLD {fold+1}/{N_FOLDS}  —  {len(tr_idx):,} train / {len(va_idx):,} val')
    print(f"{'='*60}")

    # Per-fold scaling: compute stats from training fold only
    y_tr_fold = y_raw_tr[tr_idx]
    t_means = np.nanmean(y_tr_fold, axis=0)            # (6,)
    t_stds  = np.nanstd(y_tr_fold, axis=0, ddof=1)     # (6,)
    t_stds  = np.where(t_stds < 1e-6, 1.0, t_stds)

    def scale(y):
        return (y - t_means) / t_stds  # NaN preserved

    y_tr_sc = scale(y_tr_fold)
    y_va_sc = scale(y_raw_tr[va_idx])

    ds_tr = make_dataset(smiles_arr[tr_idx].tolist(), y_tr_sc)
    ds_va = make_dataset(smiles_arr[va_idx].tolist(), y_va_sc)
    loader_tr = make_loader(ds_tr, shuffle=True)
    loader_va = make_loader(ds_va, shuffle=False)

    mpnn = make_mpnn()
    es = EarlyStopping(monitor='val_loss', patience=PATIENCE, mode='min')

    trainer = L.Trainer(
        max_epochs=MAX_EPOCHS,
        callbacks=[es],
        accelerator='cpu',
        enable_progress_bar=True,
        enable_model_summary=False,
        logger=False,
    )
    trainer.fit(mpnn, loader_tr, loader_va)

    # Un-scale task-0 (pEC50) predictions
    raw = trainer.predict(mpnn, loader_va)
    p_sc = torch.cat(raw).numpy()  # (n_val, 6)
    p_pxr = p_sc[:, TASK_PEC50] * t_stds[TASK_PEC50] + t_means[TASK_PEC50]

    y_true = y_raw_tr[va_idx, TASK_PEC50]
    m = compute_metrics(y_true, p_pxr)
    m['fold'] = fold
    m['best_epoch'] = trainer.current_epoch
    fold_metrics.append(m)
    oof_preds[va_idx] = p_pxr

    elapsed = time.time() - t_fold
    print(f'  RAE={m["RAE"]:.4f}  MAE={m["MAE"]:.4f}  Spearman={m["Spearman"]:.4f}'
          f'  best_epoch={m["best_epoch"]}  ({elapsed/60:.1f} min)')

total_min = (time.time() - t0) / 60
print(f'\nTotal CV time: {total_min:.1f} min')


FOLD 1/5  —  3,312 train / 829 val


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Loading `train_dataloader` to estimate number of stepping batches.


Output()

In [8]:
y_all = y_raw_tr[:, TASK_PEC50]
oof_rae = rae_fn(y_all, oof_preds)

cv_df = pd.DataFrame(fold_metrics)
print('5-fold scaffold CV — Chemprop 6-head auxiliary:')
print(cv_df[['fold', 'RAE', 'MAE', 'Spearman', 'best_epoch']].to_string(index=False))
print(f'\nMean fold RAE:   {cv_df["RAE"].mean():.4f} +/- {cv_df["RAE"].std():.4f}')
print(f'OOF RAE (global): {oof_rae:.4f}')
print(f'Mean Spearman:    {cv_df["Spearman"].mean():.4f}')
print(f'\nComparisons (from prior notebooks):')
print(f'  Chemprop 2-head (nb08):   0.5736')
print(f'  Chemprop 6-head (this):   {oof_rae:.4f}')

np.save(DATA_PROCESSED / 'oof_chemprop_aux.npy', oof_preds)

5-fold scaffold CV — Chemprop 6-head auxiliary:
 fold      RAE      MAE  Spearman  best_epoch
    0 0.493364 0.504667  0.779264          49
    1 0.605542 0.531939  0.692706          50
    2 0.569420 0.499185  0.715304          50
    3 0.574161 0.530997  0.736296          50
    4 0.610383 0.511546  0.697794          50

Mean fold RAE:   0.5706 +/- 0.0469
OOF RAE (global): 0.5664
Mean Spearman:    0.7243

Comparisons (from prior notebooks):
  Chemprop 2-head (nb08):   0.5736
  Chemprop 6-head (this):   0.5664


## 5. Full retrain on all training data

In [9]:
# Per-fold average convergence epoch → use as final training budget
best_epoch_mean = int(cv_df['best_epoch'].mean())
final_epochs = max(best_epoch_mean + 5, 30)
print(f'Mean best epoch across folds: {best_epoch_mean}')
print(f'Training final model for {final_epochs} epochs on all data...')

# Scale with stats from all training data
t_means_full = np.nanmean(y_raw_tr, axis=0)
t_stds_full  = np.nanstd(y_raw_tr, axis=0, ddof=1)
t_stds_full  = np.where(t_stds_full < 1e-6, 1.0, t_stds_full)

y_full_sc = (y_raw_tr - t_means_full) / t_stds_full
ds_full = make_dataset(smiles_arr.tolist(), y_full_sc)
loader_full = make_loader(ds_full, shuffle=True)

mpnn_final = make_mpnn()
trainer_final = L.Trainer(
    max_epochs=final_epochs,
    accelerator='cpu',
    enable_progress_bar=True,
    enable_model_summary=False,
    logger=False,
)
t_train = time.time()
trainer_final.fit(mpnn_final, loader_full)
print(f'Final model training time: {(time.time()-t_train)/60:.1f} min')

Mean best epoch across folds: 49
Training final model for 54 epochs on all data...


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


Loading `train_dataloader` to estimate number of stepping batches.


Output()

## 6. Predict test set

In [10]:
# Build test dataset (no labels needed for prediction)
te_dpts = [cdata.MoleculeDatapoint.from_smi(s) for s in te.smiles]
te_ds   = cdata.MoleculeDataset(te_dpts)
te_loader = make_loader(te_ds, batch_size=128, shuffle=False)

raw_te = trainer_final.predict(mpnn_final, te_loader)
p_te_sc = torch.cat(raw_te).numpy()  # (513, 6)

# Un-scale pEC50 (head 0)
te_preds = p_te_sc[:, TASK_PEC50] * t_stds_full[TASK_PEC50] + t_means_full[TASK_PEC50]

# Clip to training range ± 0.5
lo_clip = float(np.nanmin(y_raw_tr[:, TASK_PEC50])) - 0.5
hi_clip = float(np.nanmax(y_raw_tr[:, TASK_PEC50])) + 0.5
te_preds = np.clip(te_preds, lo_clip, hi_clip)

np.save(DATA_PROCESSED / 'te_chemprop_aux.npy', te_preds)

# Also report un-scaled auxiliary predictions for diagnostic
te_logp_pred = p_te_sc[:, TASK_LOGP] * t_stds_full[TASK_LOGP] + t_means_full[TASK_LOGP]
te_tpsa_pred = p_te_sc[:, TASK_TPSA] * t_stds_full[TASK_TPSA] + t_means_full[TASK_TPSA]
te_sim_pred  = p_te_sc[:, TASK_SIM]  * t_stds_full[TASK_SIM]  + t_means_full[TASK_SIM]

# Compare predicted vs actual auxiliary (test has no ground truth — use as sanity check)
te_logp_actual = physchem_te[:, 0]
te_tpsa_actual = physchem_te[:, 1]
te_sim_actual  = sim_max_te

from scipy.stats import pearsonr
print('Test auxiliary head calibration (predicted vs RDKit/actual):')
for name, pred, actual in [
    ('logP',        te_logp_pred, te_logp_actual),
    ('TPSA',        te_tpsa_pred, te_tpsa_actual),
    ('pxr_sim_max', te_sim_pred,  te_sim_actual),
]:
    mask = ~(np.isnan(pred) | np.isnan(actual))
    if mask.sum() > 10:
        r, _ = pearsonr(pred[mask], actual[mask])
        mae  = np.mean(np.abs(pred[mask] - actual[mask]))
        print(f'  {name:15s}: Pearson r={r:.3f}  MAE={mae:.3f}')

print(f'\nTest pEC50 predictions:')
print(f'  min={te_preds.min():.2f}  median={np.median(te_preds):.2f}  max={te_preds.max():.2f}')

Dropping last batch of size 1 to avoid issues with batch normalization     (dataset size = 513, batch_size = 128)


Output()

Test auxiliary head calibration (predicted vs RDKit/actual):
  logP           : Pearson r=0.949  MAE=0.268
  TPSA           : Pearson r=0.930  MAE=5.933
  pxr_sim_max    : Pearson r=0.902  MAE=0.018

Test pEC50 predictions:
  min=2.40  median=4.97  max=5.88


## 7. Save submission

In [11]:
sub = pd.DataFrame({
    'Molecule Name': te['name'].values,
    'SMILES':        te['smiles'].values,
    'pEC50':         te_preds,
})
assert len(sub) == 513 and sub['pEC50'].notna().all(), 'Submission validation failed'

out_path = SUBMISSIONS / '35_chemprop_auxiliary.csv'
sub.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'OOF RAE: {oof_rae:.4f}')
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\35_chemprop_auxiliary.csv
OOF RAE: 0.5664
count    513.000
mean       4.831
std        0.584
min        2.402
25%        4.596
50%        4.972
75%        5.250
max        5.882
Name: pEC50, dtype: float64


## Summary

| Model | CV RAE | Architecture | Notes |
|---|---|---|---|
| Chemprop 2-head (nb08) | 0.5736 | BMP+FFN, 2 tasks | pEC50 + pEC50_null |
| **Chemprop 6-head (this)** | **see above** | BMP+FFN, 6 tasks | + emax, logP, TPSA, pxr_sim |

**Saved:**
- `data/processed/oof_chemprop_aux.npy` — OOF pEC50 predictions
- `data/processed/te_chemprop_aux.npy` — test pEC50 predictions
- `submissions/35_chemprop_auxiliary.csv`

**Next:** Include in grand ensemble (nb25+) with OOF-calibrated weights.